# RAVE Dissertation Experiments
**Mihir Apte — MSc Data Science, TCD**

Run cells top to bottom. Select **A100 GPU** in Runtime > Change runtime type first.

**Methods:** Baseline | Semantic v1 (greedy) | Semantic v2 (K-means) | Multi-ControlNet + FreeU

In [ ]:
# Cell 1 - Check GPU
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("NO GPU - change runtime type to A100 before continuing")


In [ ]:
# Cell 2 - Clone repo
import os
REPO = "/content/dissertation-mihir"
if os.path.exists(REPO):
    print("Repo exists - pulling latest...")
    !cd {REPO} && git pull origin main
else:
    !git clone https://github.com/MihirApte/dissertation-mihir.git {REPO}
os.chdir(REPO)
print("Done.")


In [ ]:
# Cell 3 - Install packages
!pip install -q diffusers transformers accelerate omegaconf einops open_clip_torch scikit-learn Pillow opencv-python imageio imageio-ffmpeg yt-dlp
!pip install -q git+https://github.com/openai/CLIP.git
print("All packages installed.")


In [ ]:
# Cell 4 - Patch basicsr (torchvision 0.17+ removed functional_tensor)
import glob
matches = glob.glob("/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py")
if matches:
    with open(matches[0]) as f: content = f.read()
    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"
    if old in content:
        with open(matches[0], "w") as f: f.write(content.replace(old, new))
        print("basicsr patched")
    else:
        print("basicsr already patched")
else:
    print("basicsr not found - skipping")


In [ ]:
# Cell 5 - Patch ZoeDepth
# Fix 1: use GPU (not CPU)
# Fix 2: strict=False to handle newer timm key mismatches
zoe_path = "/content/dissertation-mihir/annotator/zoe/__init__.py"
with open(zoe_path) as f: content = f.read()

# Fix 1: revert any CPU patching
content = content.replace('self.model.to("cpu")', "self.model.to(self.device)")
content = content.replace('torch.from_numpy(image_depth).float().to("cpu")',
                          "torch.from_numpy(image_depth).float().to(self.device)")

# Fix 2: add strict=False to load_state_dict
old = "model.load_state_dict(torch.load(modelpath, map_location=model.device)['model'])"
new = "model.load_state_dict(torch.load(modelpath, map_location=model.device)['model'], strict=False)"
if old in content:
    content = content.replace(old, new)
    print("ZoeDepth strict=False patch applied")
else:
    print("ZoeDepth strict=False already patched")

with open(zoe_path, "w") as f: f.write(content)
print("ZoeDepth patched: GPU mode + strict=False")


In [ ]:
# Cell 6 - Upload truck.mp4 from your computer
# You only need to upload truck.mp4 here.
# The other 3 videos (shanghai, street, dog) are downloaded automatically in Cell 7.
import os, shutil
from google.colab import files

VIDEO_DIR = "/content/dissertation-mihir/data/mp4_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

truck_path = f"{VIDEO_DIR}/truck.mp4"
if os.path.exists(truck_path):
    size = os.path.getsize(truck_path) / 1e6
    print(f"truck.mp4 already present ({size:.1f} MB) - skipping upload")
else:
    print("Select truck.mp4 from your computer...")
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, truck_path)
        print(f"Saved truck.mp4 ({os.path.getsize(truck_path)/1e6:.1f} MB)")


In [ ]:
# Cell 7 - Download shanghai, street, dog from YouTube (first 10 seconds each)
import os
VIDEO_DIR = "/content/dissertation-mihir/data/mp4_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

videos = {
    "shanghai.mp4": "https://youtu.be/n5cW4FpGvhI",
    "street.mp4":   "https://youtu.be/1XIOmKGjgho",
    "dog.mp4":      "https://youtu.be/7LycCv0PIBo",
}

for filename, url in videos.items():
    out = f"{VIDEO_DIR}/{filename}"
    if os.path.exists(out):
        print(f"{filename} already exists - skipping")
        continue
    print(f"Downloading {filename}...")
    !yt-dlp --download-sections "*0:00-0:10" \n        -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \n        --merge-output-format mp4 \n        -o "{out}" "{url}"

# Verify all 4
print("
Video check:")
for v in ["truck.mp4", "shanghai.mp4", "street.mp4", "dog.mp4"]:
    path = f"{VIDEO_DIR}/{v}"
    if os.path.exists(path):
        print(f"  OK  {v}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  MISSING  {v}")


In [ ]:
# Cell 8 - Run ALL experiments
# Runs: baseline, semantic v1 (greedy), semantic v2 (kmeans), multi-controlnet
# for all 4 videos. Expected time: ~5-6 hours on A100.
# Do NOT close this tab while running.
import os
os.chdir("/content/dissertation-mihir")
!bash run_all_experiments.sh 2>&1


In [ ]:
# Cell 9 - Show full metrics table
import os
os.chdir("/content/dissertation-mihir")
results_file = "results/metrics_all_methods.txt"
if os.path.exists(results_file):
    with open(results_file) as f:
        print(f.read())
else:
    print("Not found - running metrics now...")
    !python3 compute_metrics_all.py --device cuda


In [ ]:
# Cell 10 - Download all results (GIFs + metrics) as zip
import shutil
from google.colab import files
shutil.make_archive("/content/rave_results", "zip", "/content/dissertation-mihir/results")
files.download("/content/rave_results.zip")
print("Downloading rave_results.zip...")
print("If you get path-too-long error on Windows extraction, use 7-zip.")
